# Sample descriptives

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

from data_preparation.config import connect_db

In [ ]:
# Import shared database configuration
ROOT_DIR = Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "data_preparation"))

# Paths
OUT_DIR = ROOT_DIR / "results" / "descriptives"
DB_NAME = ROOT_DIR / "../insurance.duckdb"
unique_ids_main_analysis = ROOT_DIR / "../unique_ids_final.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load analysis cohort
main_analysis_ids = pd.read_csv(unique_ids_main_analysis)

# Connect to DuckDB using shared configuration
con = connect_db(str(DB_NAME))

# Analysis-specific DuckDB settings
TEMP_DIR = ROOT_DIR / "tmp" / "duckdb"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

con.execute("SET memory_limit = '48GB'")
con.execute("SET threads = 8")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET max_temp_directory_size = '14GiB'")
con.execute("SET preserve_insertion_order = false")


In [ ]:
# unique_ids holds final analysed sample from diff-in-diff

# Count original rows
n_uid_raw = con.execute(
    "SELECT COUNT(*) FROM read_csv_auto(?)",
    [str(unique_ids_main_analysis)]
).fetchone()[0]

# Create temporary table with unique IDs
con.execute("""
    CREATE OR REPLACE TEMP TABLE uid AS
    SELECT DISTINCT eth_id
    FROM read_csv_auto(?)
""", [str(unique_ids_main_analysis)])

# Count distinct IDs
n_uid = con.execute(
    "SELECT COUNT(*) FROM uid"
).fetchone()[0]

print(f"unique_ids: {n_uid_raw:,} raw; {n_uid:,} after DISTINCT")

In [ ]:
# Lancet-style number formatting
THIN = '\u2009'
MINUS = '\u2212'

def fn(n):
    """Integer: thin-space thousands for ≥5 digits, closed up for 4 digits."""
    n = int(round(n))
    s = f"{abs(n):d}"
    if len(s) >= 5:
        s = f"{abs(n):,}".replace(',', THIN)
    return (MINUS if n < 0 else '') + s

def fd(x, d=1):
    """Decimal with mid-dot and true minus sign."""
    s = f"{abs(x):.{d}f}".replace('.', '·')
    return (MINUS if x < 0 else '') + s

def tick_fmt(d=0, suffix=''):
    return mticker.FuncFormatter(lambda x, _: fd(x, d) + suffix)

# Define comparison "general population"

All insured adults aged ≥18 years with mandatory insurance coverage at some time during the study period, regardless of app use

In [ ]:
# Overall population  
# 1. ≥1 inpatient or outpatient claim, 2018–2024
# 2. ∩ mandatory (KVG) contract overlapping 2018–2024
# 3. ∩ age ≥ 18 in at least one persons record (ever adult)

con.execute("""
    CREATE OR REPLACE TEMP TABLE pop_steps AS
    WITH claimants AS (
        SELECT eth_id FROM inpatient
        WHERE TRY_CAST(behandlungsbeginn AS DATE) BETWEEN DATE '2018-01-01' AND DATE '2024-12-31'
        UNION
        SELECT eth_id FROM outpatient
        WHERE TRY_CAST(behandlungsbeginn AS DATE) BETWEEN DATE '2018-01-01' AND DATE '2024-12-31'
    ),
    has_obligatory AS (
        SELECT DISTINCT eth_id
        FROM contracts_obligatory
        WHERE TRY_CAST(beginndatum AS DATE) <= DATE '2024-12-31'
          AND COALESCE(TRY_CAST(endedatum AS DATE), DATE '9999-12-31') >= DATE '2018-01-01'
    ),
    ever_adult AS (
        SELECT eth_id
        FROM persons
        GROUP BY eth_id
        HAVING MAX(TRY_CAST(alter_kunde AS INTEGER)) >= 18
    )
    SELECT c.eth_id,
           (o.eth_id IS NOT NULL)                          AS s2_obligatory,
           (o.eth_id IS NOT NULL AND a.eth_id IS NOT NULL) AS s3_adult
    FROM claimants c
    LEFT JOIN has_obligatory o ON o.eth_id = c.eth_id
    LEFT JOIN ever_adult     a ON a.eth_id = c.eth_id
""")

con.execute("""
    CREATE OR REPLACE TEMP TABLE filtered_pop AS
    SELECT eth_id FROM pop_steps WHERE s3_adult
""")

n1, n2, n3 = con.execute("""
    SELECT COUNT(*), COUNT(*) FILTER (s2_obligatory), COUNT(*) FILTER (s3_adult)
    FROM pop_steps
""").fetchone()

print(f"1. ≥1 claim 2018–24:                  {n1:>10,}   (expected 2,129,205)  {'✓' if n1 == 2_129_205 else '✗'}")
print(f"2. + mandatory contract 2018–24:      {n2:>10,}   (expected 2,030,090)  {'✓' if n2 == 2_030_090 else '✗'}")
print(f"   removed (no mandatory contract):   {n1 - n2:>10,}")
print(f"3. + 18+ at some point:               {n3:>10,}")
print(f"   removed (never 18+): {n2 - n3:>10,}")
n_filtered_pop = n3

## Sample description diff-in-diff analysed sample vs. general population

In [ ]:
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

LBL_DID = 'Analysis sample'
LBL_POP = 'Overall population'
PCG_YEARS = None          # e.g. (2018, 2024) to restrict "ever" to that window

# ── 1. Income index ───────────────────────────────────────────────────────────
# Groups (overlapping by design — overall population includes adopters):
#   Analysis sample    = uid           (N = 100,281)
#   Overall population = filtered_pop  (N = 1,623,841)
# Income = latest non-missing einkommensindex per person.

income_all = con.execute(f"""
    WITH latest_income AS (
        SELECT DISTINCT ON (eth_id)
            eth_id,
            TRY_CAST(einkommensindex AS DOUBLE) AS income_index
        FROM income_index
        WHERE TRY_CAST(einkommensindex AS DOUBLE) IS NOT NULL
        ORDER BY eth_id, TRY_CAST(endedatum AS DATE) DESC NULLS FIRST
    ),
    grp AS (
        SELECT eth_id, '{LBL_DID}' AS group_label FROM uid
        UNION ALL
        SELECT eth_id, '{LBL_POP}' AS group_label FROM filtered_pop
    )
    SELECT g.eth_id, g.group_label, li.income_index
    FROM grp g
    LEFT JOIN latest_income li ON li.eth_id = g.eth_id
""").fetchdf()

inc_n = (income_all.groupby('group_label')['income_index']
         .agg(N='size', n_income='count'))
inc_n['n_missing'] = inc_n['N'] - inc_n['n_income']
print(inc_n)

assert inc_n.loc[LBL_DID, 'N'] == 100_281,   inc_n.loc[LBL_DID, 'N']
assert inc_n.loc[LBL_POP, 'N'] == 1_623_841, inc_n.loc[LBL_POP, 'N']

income = income_all.dropna(subset=['income_index'])

# ── 2. PCG columns ────────────────────────────────────────────────────────────
pcg_cols = con.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'pcg_wide'
      AND column_name LIKE 'pcg_%'
      AND column_name NOT IN ('pcg_jahr', 'pcg_versionsnummer')
    ORDER BY column_name
""").fetchdf()['column_name'].tolist()
print(f"PCG columns found: {len(pcg_cols)}")

pcg_max_cols  = ",\n    ".join(f"MAX(COALESCE(TRY_CAST({c} AS INTEGER), 0)) AS {c}" for c in pcg_cols)
pcg_fill_cols = ",\n    ".join(f"COALESCE(pp.{c}, 0) AS {c}" for c in pcg_cols)
year_filter = (f"AND TRY_CAST(pcg_jahr AS INTEGER) BETWEEN {PCG_YEARS[0]} AND {PCG_YEARS[1]}"
               if PCG_YEARS else "")

# ── 3. PCG ever-positive per person ───────────────────────────────────────────
# pcg_wide has rows for everyone (incl. all-zero rows), so "no row" = missing data.
# Prevalence denominator = group members with ≥1 pcg_wide row.
def pcg_for_group(group_table):
    return con.execute(f"""
        WITH pcg_person AS (
            SELECT eth_id, {pcg_max_cols}
            FROM pcg_wide
            WHERE eth_id IN (SELECT eth_id FROM {group_table})
              {year_filter}
            GROUP BY eth_id
        )
        SELECT g.eth_id,
               (pp.eth_id IS NOT NULL) AS has_pcg_record,
               {pcg_fill_cols}
        FROM {group_table} g
        LEFT JOIN pcg_person pp ON pp.eth_id = g.eth_id
    """).fetchdf()

pcg_did = pcg_for_group('uid')
pcg_pop = pcg_for_group('filtered_pop')

n_did, n_pop = len(pcg_did), len(pcg_pop)
assert n_did == 100_281,   n_did
assert n_pop == 1_623_841, n_pop

pcg_did_rec = pcg_did[pcg_did['has_pcg_record']]
pcg_pop_rec = pcg_pop[pcg_pop['has_pcg_record']]
n_did_rec, n_pop_rec = len(pcg_did_rec), len(pcg_pop_rec)
n_did_miss, n_pop_miss = n_did - n_did_rec, n_pop - n_pop_rec

print(f"{LBL_DID}: N={n_did:,} | PCG data {n_did_rec:,} ({100*n_did_rec/n_did:.1f}%) | missing {n_did_miss:,} ({100*n_did_miss/n_did:.1f}%)")
print(f"{LBL_POP}: N={n_pop:,} | PCG data {n_pop_rec:,} ({100*n_pop_rec/n_pop:.1f}%) | missing {n_pop_miss:,} ({100*n_pop_miss/n_pop:.1f}%)")

# ── 3b. Swiss PCG label map ───────────────────────────────────────────────────
pcg_labels = {
    'pcg_abh': 'Addiction (excl. Opioids)',
    'pcg_adh': 'ADHD',
    'pcg_aik': 'IBD',
    'pcg_alz': "Alzheimer's / Dementia",
    'pcg_ast': 'Asthma',
    'pcg_bsr': 'Bone / Skeletal Disorders',
    'pcg_car': 'Cardiac Arrhythmia',
    'pcg_cop': 'COPD',
    'pcg_dep': 'Depression',
    'pcg_dm1': 'Type 1 Diabetes',
    'pcg_dm2': 'Type 2 Diabetes',
    'pcg_dm2hyp': 'Type 2 Diabetes with Hypertension',
    'pcg_epi': 'Epilepsy',
    'pcg_gla': 'Glaucoma',
    'pcg_hch': 'High Cholesterol',
    'pcg_hiv': 'HIV',
    'pcg_kho': 'Coronary Heart Disease',
    'pcg_kre': 'Cancer',
    'pcg_krk': 'Cancer (other)',
    'pcg_mcr': 'Macular Degeneration',
    'pcg_msk': 'Musculoskeletal Disorders',
    'pcg_nie': 'Kidney Disease',
    'pcg_pah': 'Pulmonary Arterial Hypertension',
    'pcg_par': "Parkinson's",
    'pcg_pso': 'Psoriasis',
    'pcg_psy': 'Severe Psychiatric Disorders',
    'pcg_rhe': 'Rheumatoid Arthritis',
    'pcg_smc': 'Severe Mental Conditions',
    'pcg_smn': 'Neuropathic Pain',
    'pcg_thy': 'Thyroid Disorders',
    'pcg_tra': 'Transplantation',
    'pcg_was': 'Growth Disorders',
    'pcg_zfp': 'Cystic Fibrosis / Pancreatic Enzymes',
    'pcg_zns': 'CNS Disorders (excl. MS)',
}

# ── 4. Top 10 PCGs by overall-population prevalence ───────────────────────────
cnt_did = pcg_did_rec[pcg_cols].sum()
cnt_pop = pcg_pop_rec[pcg_cols].sum()
prev_did_all = cnt_did / n_did_rec * 100
prev_pop_all = cnt_pop / n_pop_rec * 100

TOP_N = 15
top_pcgs = prev_pop_all.sort_values(ascending=False).head(TOP_N).index.tolist()

unmapped = [p for p in top_pcgs if p not in pcg_labels]
if unmapped:
    print(f"⚠ No label for: {unmapped}")

df_pcg = pd.DataFrame({
    'pcg':      top_pcgs,
    'n_did':    [int(cnt_did[p]) for p in top_pcgs],
    'prev_did': [prev_did_all[p] for p in top_pcgs],
    'n_pop':    [int(cnt_pop[p]) for p in top_pcgs],
    'prev_pop': [prev_pop_all[p] for p in top_pcgs],
})

df_pcg['label'] = df_pcg['pcg'].map(lambda x: pcg_labels.get(x, x))
df_pcg = df_pcg.sort_values('prev_pop', ascending=True)
print(df_pcg[['label', 'n_did', 'prev_did', 'n_pop', 'prev_pop']].round(2).to_string(index=False))

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 1 — Income index distribution
# ═════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))

colors = {LBL_DID: '#E84A5F', LBL_POP: '#2B4590'}
alphas = {LBL_DID: 0.45,      LBL_POP: 0.35}

def n_label(grp):
    r = inc_n.loc[grp]
    return (f"N={fn(r.N)}; income available n={fn(r.n_income)} "
            f"(missing {fn(r.n_missing)}, {fd(100*r.n_missing/r.N)}%)")

summary_lines = []
for grp in [LBL_DID, LBL_POP]:
    vals = income.loc[income['group_label'] == grp, 'income_index']
    mn, med = vals.mean(), vals.median()

    ax.hist(vals, bins=80, density=True, color=colors[grp], alpha=alphas[grp])
    ax.axvline(mn,  color=colors[grp], lw=2.0, ls='-')
    ax.axvline(med, color=colors[grp], lw=2.0, ls=':')

    summary_lines.append(
        f"{grp}\n({n_label(grp)})\n"
        f"mean={fd(mn, 3)}  |  median={fd(med, 3)}  |  sd={fd(vals.std(), 3)}"
    )

legend_handles = [
    Line2D([0], [0], color=colors[LBL_DID], lw=2.5, alpha=0.9,
           label=f"{LBL_DID} (N={fn(inc_n.loc[LBL_DID, 'N'])})"),
    Line2D([0], [0], color=colors[LBL_POP], lw=2.5, alpha=0.75,
           label=f"{LBL_POP} (N={fn(inc_n.loc[LBL_POP, 'N'])})"),
    Line2D([0], [0], color='gray', lw=2.0, ls='-', label='Mean'),
    Line2D([0], [0], color='gray', lw=2.0, ls=':', label='Median'),
]
ax.legend(handles=legend_handles, fontsize=10, ncol=2, loc='upper right')

ax.text(0.99, 0.75, '\n'.join(summary_lines),
        transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right',
        multialignment='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85))

ax.xaxis.set_major_formatter(tick_fmt(2))
ax.yaxis.set_major_formatter(tick_fmt(2))
ax.set_xlabel('Income index', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/descriptives/income_distribution_did_vs_pop.png', dpi=150, bbox_inches='tight')
plt.show()

# ═════════════════════════════════════════════════════════════════════════════
# PLOT 2 — PCG prevalence (among those with PCG data)
# ═════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 8.5))
y, h = np.arange(len(df_pcg)), 0.38

b1 = ax.barh(y + h/2, df_pcg['prev_did'], h,
             label=(f'{LBL_DID} (N={fn(n_did)}; PCG data n={fn(n_did_rec)}, '
                    f'missing {fn(n_did_miss)}, {fd(100*n_did_miss/n_did)}%)'),
             color=colors[LBL_DID], alpha=0.85)
b2 = ax.barh(y - h/2, df_pcg['prev_pop'], h,
             label=(f'{LBL_POP} (N={fn(n_pop)}; PCG data n={fn(n_pop_rec)}, '
                    f'missing {fn(n_pop_miss)}, {fd(100*n_pop_miss/n_pop)}%)'),
             color=colors[LBL_POP], alpha=0.65)

for bars, col, ncol in [(b1, colors[LBL_DID], 'n_did'), (b2, colors[LBL_POP], 'n_pop')]:
    for bar, n in zip(bars, df_pcg[ncol]):
        w = bar.get_width()
        ax.text(w + 0.15, bar.get_y() + bar.get_height()/2,
                f'{fd(w)}% (n={fn(n)})', va='center', fontsize=8.5, color=col)

ax.set_yticks(y)
ax.set_yticklabels(df_pcg['label'], fontsize=11)
ax.set_xlabel('Prevalence (% of group with PCG data)', fontsize=12)
ax.xaxis.set_major_formatter(tick_fmt(0, '%'))
ax.set_xlim(0, df_pcg[['prev_did', 'prev_pop']].max().max() * 1.35)
ax.legend(fontsize=9.5, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/descriptives/pcg_prevalence_did_vs_pop.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nDone. Saved: income_distribution_did_vs_pop.png, pcg_prevalence_did_vs_pop.png")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Combined characteristics table: DiD sample vs. filtered overall population
# (filtered_pop = 18+ at any persons-record, ever held a contracts_obligatory row;
#  built in the earlier cell. DiD sample = uid, left as-is.)
# ═════════════════════════════════════════════════════════════════════════════

# ── A. Build per-person coverage-years denominator (for "avg yearly" metrics) ──
# Years of mandatory (KVG) coverage per person, used as the denominator for
# avg yearly costs/claims. Counted as distinct calendar years touched by any
# contracts_obligatory interval [beginndatum, endedatum].
con.execute("""
    CREATE OR REPLACE TEMP TABLE coverage_years AS
    WITH bounds AS (
        SELECT
            eth_id,
            TRY_CAST(beginndatum AS DATE) AS start_dt,
            TRY_CAST(endedatum   AS DATE) AS end_dt
        FROM contracts_obligatory
        WHERE TRY_CAST(beginndatum AS DATE) IS NOT NULL
    ),
    yrs AS (
        SELECT eth_id, UNNEST(GENERATE_SERIES(YEAR(start_dt), YEAR(COALESCE(end_dt, start_dt)))) AS yr
        FROM bounds
    )
    SELECT eth_id, COUNT(DISTINCT yr) AS n_coverage_years
    FROM yrs
    GROUP BY eth_id
""")

# ── B. Per-person cost/claim totals (KVG-only, all years, both groups together) ─
con.execute("""
    CREATE OR REPLACE TEMP TABLE person_claims AS
    WITH ip AS (
        SELECT
            eth_id,
            COUNT(*) AS n_inpatient_claims,
            SUM(TRY_CAST(betrag AS DOUBLE)) AS inpatient_cost
        FROM inpatient
        WHERE UPPER(TRIM(gesetz)) = 'KVG'
        GROUP BY eth_id
    ),
    op AS (
        SELECT
            eth_id,
            COUNT(*) AS n_outpatient_claims,
            SUM(TRY_CAST(betrag AS DOUBLE)) AS outpatient_cost
        FROM outpatient
        WHERE UPPER(TRIM(gesetz)) = 'KVG'
        GROUP BY eth_id
    )
    SELECT
        COALESCE(ip.eth_id, op.eth_id) AS eth_id,
        COALESCE(ip.n_inpatient_claims, 0)  AS n_inpatient_claims,
        COALESCE(ip.inpatient_cost, 0.0)    AS inpatient_cost,
        COALESCE(op.n_outpatient_claims, 0) AS n_outpatient_claims,
        COALESCE(op.outpatient_cost, 0.0)   AS outpatient_cost,
        COALESCE(ip.inpatient_cost, 0.0) + COALESCE(op.outpatient_cost, 0.0) AS total_claim_cost,
        COALESCE(ip.n_inpatient_claims, 0) + COALESCE(op.n_outpatient_claims, 0) AS total_claims
    FROM ip
    FULL OUTER JOIN op ON ip.eth_id = op.eth_id
""")

# ── C. Per-person franchise (latest value per person, from contracts_obligatory) ─
con.execute("""
    CREATE OR REPLACE TEMP TABLE person_franchise AS
    SELECT DISTINCT ON (eth_id)
        eth_id,
        TRY_CAST(franchise_betrag AS DOUBLE) AS franchise
    FROM contracts_obligatory
    WHERE TRY_CAST(franchise_betrag AS DOUBLE) IS NOT NULL
    ORDER BY eth_id, TRY_CAST(endedatum AS DATE) DESC
""")

# ── D. Per-person PCG count (ever, across all years) ────────────────────────────
pcg_count_all = con.execute(f"""
    SELECT p.eth_id, {pcg_max_cols}
    FROM pcg_wide p
    GROUP BY p.eth_id
""").fetchdf()
pcg_count_all['n_pcgs'] = pcg_count_all[pcg_cols].sum(axis=1)
pcg_count_all = pcg_count_all[['eth_id', 'n_pcgs']]

# ── E. Pull everything into one Python frame per group ──────────────────────────
def build_group_table(eth_ids_query):
    """eth_ids_query: SQL returning a single eth_id column defining the group."""
    demo = con.execute(f"""
        WITH grp AS ({eth_ids_query}),
        latest_person AS (
            SELECT DISTINCT ON (eth_id)
                eth_id,
                TRY_CAST(alter_kunde AS INTEGER) AS age,
                geschlecht,
                kanton
            FROM persons
            ORDER BY eth_id, TRY_CAST(endedatum AS DATE) DESC
        ),
        latest_income AS (
            SELECT DISTINCT ON (eth_id)
                eth_id, einkommensindex AS income_index
            FROM income_index
            ORDER BY eth_id, CAST(endedatum AS DATE) DESC
        )
        SELECT
            g.eth_id,
            lp.age,
            lp.geschlecht,
            lp.kanton,
            li.income_index,
            cy.n_coverage_years,
            pc.n_inpatient_claims, pc.inpatient_cost,
            pc.n_outpatient_claims, pc.outpatient_cost,
            pc.total_claim_cost, pc.total_claims,
            pf.franchise
        FROM grp g
        LEFT JOIN latest_person lp ON lp.eth_id = g.eth_id
        LEFT JOIN latest_income li ON li.eth_id = g.eth_id
        LEFT JOIN coverage_years cy ON cy.eth_id = g.eth_id
        LEFT JOIN person_claims pc ON pc.eth_id = g.eth_id
        LEFT JOIN person_franchise pf ON pf.eth_id = g.eth_id
    """).fetchdf()

    demo = demo.merge(pcg_count_all, on='eth_id', how='left')
    demo['n_pcgs'] = demo['n_pcgs'].fillna(0)

    # avg yearly metrics: divide by coverage years; guard against 0/NULL years
    safe_years = demo['n_coverage_years'].replace(0, np.nan)
    demo['avg_yearly_total_cost']      = demo['total_claim_cost'] / safe_years
    demo['avg_yearly_inpatient_cost']  = demo['inpatient_cost'] / safe_years
    demo['avg_yearly_outpatient_cost'] = demo['outpatient_cost'] / safe_years
    demo['avg_yearly_total_claims']      = demo['total_claims'] / safe_years
    demo['avg_yearly_inpatient_claims']  = demo['n_inpatient_claims'] / safe_years
    demo['avg_yearly_outpatient_claims'] = demo['n_outpatient_claims'] / safe_years

    return demo

did_tbl = build_group_table("SELECT eth_id FROM uid")
pop_tbl = build_group_table("SELECT eth_id FROM filtered_pop")

# ── F. Summarize into one combined table ─────────────────────────────────────────
def fmt_mean_sd(s):
    s = s.dropna()
    return f"{s.mean():.2f} ({s.std():.2f})"

def fmt_median_minmax(s):
    s = s.dropna()
    return f"{s.median():.1f} [{s.min():.0f}\u2013{s.max():.0f}]"

def fmt_median_iqr(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return f"{s.median():.2f} ({q1:.2f}\u2013{q3:.2f})"

def pct(n, d):
    return f"{100*n/d:.1f}%" if d else "n/a"

def summarize(tbl, n_total):
    rows = {}

    # Age
    age = tbl['age'].dropna()
    rows['Age, mean (SD)']        = fmt_mean_sd(tbl['age'])
    rows['Age, median [min\u2013max]'] = fmt_median_minmax(tbl['age'])

    # Sex
    n_f = (tbl['geschlecht'] == 'F').sum()
    n_m = (tbl['geschlecht'] == 'M').sum()
    n_other = n_total - n_f - n_m
    rows['Sex, female, %'] = pct(n_f, n_total)
    rows['Sex, male, %']   = pct(n_m, n_total)
    rows['Sex, other/missing, %'] = pct(n_other, n_total)

    # Income
    rows['Income index, median (IQR)'] = fmt_median_iqr(tbl['income_index'])

    # Canton: top 5 + "other", as % of group
    canton_counts = tbl['kanton'].value_counts(dropna=True)
    canton_pct = (canton_counts / n_total * 100).round(1)
    top5 = canton_pct.head(5)
    other_pct = canton_pct.iloc[5:].sum() if len(canton_pct) > 5 else 0.0
    missing_canton_pct = 100 * tbl['kanton'].isna().sum() / n_total
    canton_str = ", ".join([f"{idx} {val:.1f}%" for idx, val in top5.items()])
    canton_str += f", Other {other_pct:.1f}%, Missing {missing_canton_pct:.1f}%"
    rows['Canton (top 5, % of group)'] = canton_str

    # Costs (avg yearly)
    rows['Avg yearly total cost (CHF), mean (SD)']      = fmt_mean_sd(tbl['avg_yearly_total_cost'])
    rows['Avg yearly inpatient cost (CHF), mean (SD)']  = fmt_mean_sd(tbl['avg_yearly_inpatient_cost'])
    rows['Avg yearly outpatient cost (CHF), mean (SD)'] = fmt_mean_sd(tbl['avg_yearly_outpatient_cost'])

    # Claim counts (avg yearly)
    rows['Avg yearly total claims, mean (SD)']      = fmt_mean_sd(tbl['avg_yearly_total_claims'])
    rows['Avg yearly inpatient claims, mean (SD)']  = fmt_mean_sd(tbl['avg_yearly_inpatient_claims'])
    rows['Avg yearly outpatient claims, mean (SD)'] = fmt_mean_sd(tbl['avg_yearly_outpatient_claims'])

    # Franchise
    rows['Franchise (CHF), mean (SD)']   = fmt_mean_sd(tbl['franchise'])
    rows['Franchise (CHF), median']      = f"{tbl['franchise'].median():.0f}"

    # PCGs
    rows['Number of PCGs ever, mean (SD)']   = fmt_mean_sd(tbl['n_pcgs'])
    rows['Number of PCGs ever, median']      = f"{tbl['n_pcgs'].median():.1f}"
    rows['\u22651 PCG, %'] = pct((tbl['n_pcgs'] >= 1).sum(), n_total)

    return rows

n_did = len(did_tbl)
n_pop = len(pop_tbl)

did_summary = summarize(did_tbl, n_did)
pop_summary = summarize(pop_tbl, n_pop)

combined = pd.DataFrame({
    f'DiD sample (N={n_did:,})': did_summary,
    f'Overall population, filtered (N={n_pop:,})': pop_summary,
})

print(combined.to_string())
combined.to_csv('results/descriptives/characteristics_table_did_vs_pop.csv')
print("\nSaved: results/descriptives/characteristics_table_did_vs_pop.csv")

## Characteristics available for adopters only

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Adopters-only (DiD sample = uid) summary:
#   - BMI: mean/median/SD/min-max, WHO category counts + %, missing
#   - Daily steps: mean/SD per person, counting only days with a recorded value
#   - Sessions/month: mean/SD per person, counting only months with >=1 transaction/session
# ═════════════════════════════════════════════════════════════════════════════

# ── 0. ID mapping for the DiD sample ────────────────────────────────────────────
con.execute("""
    CREATE OR REPLACE TEMP TABLE adopter_ids AS
    SELECT DISTINCT eth_id
    FROM uid
""")

n_adopters = con.execute("SELECT COUNT(DISTINCT eth_id) FROM adopter_ids").fetchone()[0]
print(f"Adopters (DiD sample): {n_adopters:,}")

# ═════════════════════════════════════════════════════════════════════════════
# 1. BMI
# ═════════════════════════════════════════════════════════════════════════════
# One representative BMI per person: most recent non-null bmi value from
# claims_person_month_summary.
bmi_df = con.execute("""
    WITH latest_bmi AS (
        SELECT DISTINCT ON (eth_id)
            eth_id,
            bmi
        FROM claims_person_month_summary
        WHERE bmi IS NOT NULL
        ORDER BY eth_id, claim_month DESC
    )
    SELECT a.eth_id, lb.bmi
    FROM adopter_ids a
    LEFT JOIN latest_bmi lb ON lb.eth_id = a.eth_id
""").fetchdf()

bmi_vals = bmi_df['bmi'].dropna()
n_bmi_missing = bmi_df['bmi'].isna().sum()

print("\n--- BMI ---")
print(f"N with BMI: {len(bmi_vals):,} | Missing: {n_bmi_missing:,} ({100*n_bmi_missing/n_adopters:.1f}%)")
print(f"Mean: {bmi_vals.mean():.2f} | Median: {bmi_vals.median():.2f} | SD: {bmi_vals.std():.2f}")
bmi_q1, bmi_q3 = bmi_vals.quantile(0.25), bmi_vals.quantile(0.75)
print(f"IQR: {bmi_q1:.1f}\u2013{bmi_q3:.1f}")

# WHO BMI categories
def who_category(b):
    if pd.isna(b):
        return 'Missing'
    elif b < 18.5:
        return 'Underweight (<18.5)'
    elif b < 25:
        return 'Normal (18.5-24.9)'
    elif b < 30:
        return 'Overweight (25-29.9)'
    else:
        return 'Obese (>=30)'

bmi_df['who_cat'] = bmi_df['bmi'].apply(who_category)
cat_order = ['Underweight (<18.5)', 'Normal (18.5-24.9)', 'Overweight (25-29.9)', 'Obese (>=30)', 'Missing']
cat_counts = bmi_df['who_cat'].value_counts().reindex(cat_order, fill_value=0)
cat_pct = (cat_counts / n_adopters * 100).round(1)

bmi_category_table = pd.DataFrame({'N': cat_counts, '%': cat_pct})
print("\nWHO BMI categories:")
print(bmi_category_table.to_string())

# ═════════════════════════════════════════════════════════════════════════════
# 2. Daily steps (per person, mean/SD across days WITH a recorded value)
# ═════════════════════════════════════════════════════════════════════════════
print("\nDATA_TYPE values in a365_health_activities:")
print(con.execute("SELECT DISTINCT DATA_TYPE FROM a365_health_activities").fetchdf())

steps_per_person = con.execute("""
    SELECT
        a.eth_id,
        AVG(TRY_CAST(s.VALUE AS DOUBLE)) AS mean_daily_steps,
        STDDEV(TRY_CAST(s.VALUE AS DOUBLE)) AS sd_daily_steps,
        COUNT(*) AS n_days_recorded
    FROM adopter_ids a
    INNER JOIN a365_health_activities s
        ON CAST(s.CSS_MARKETING_ID AS VARCHAR) = CAST(a.eth_id AS VARCHAR)
    WHERE LOWER(TRIM(s.DATA_TYPE)) = 'steps'
      AND TRY_CAST(s.VALUE AS DOUBLE) IS NOT NULL
    GROUP BY a.eth_id
""").fetchdf()

n_steps_persons = len(steps_per_person)
print("\n--- Daily steps (per person, days with recorded value only) ---")
print(f"N persons with any recorded step-day: {n_steps_persons:,} "
      f"({100*n_steps_persons/n_adopters:.1f}% of adopters)")
print(f"Person-level mean daily steps  — mean across persons: {steps_per_person['mean_daily_steps'].mean():.1f}, "
      f"SD across persons: {steps_per_person['mean_daily_steps'].std():.1f}")
print("(i.e. each person's own mean daily step count is computed first over their recorded days, "
      "then we summarize the distribution of those person-level means across adopters)")

# ═════════════════════════════════════════════════════════════════════════════
# 3. Sessions per month (only counting months with >=1 session) — mean & SD
# ═════════════════════════════════════════════════════════════════════════════
sessions_per_person_month = con.execute("""
    SELECT
        a.eth_id,
        DATE_TRUNC('month', CAST(s.DATE_TIME AS TIMESTAMP)) AS session_month,
        COUNT(*) AS n_sessions
    FROM adopter_ids a
    INNER JOIN a365_health_sessions s
        ON CAST(s.CSS_MARKETING_ID AS VARCHAR) = CAST(a.eth_id AS VARCHAR)
    WHERE s.DATE_TIME IS NOT NULL
    GROUP BY a.eth_id, DATE_TRUNC('month', CAST(s.DATE_TIME AS TIMESTAMP))
""").fetchdf()

# per-person mean sessions/month, computed only over that person's active months
person_avg_sessions = (
    sessions_per_person_month
    .groupby('eth_id')['n_sessions']
    .mean()
    .reset_index(name='avg_sessions_per_active_month')
)

n_session_persons = len(person_avg_sessions)
print("\n--- Sessions per month (active months only, i.e. months with >=1 session) ---")
print(f"N persons with any session: {n_session_persons:,} "
      f"({100*n_session_persons/n_adopters:.1f}% of adopters)")
print(f"Mean of person-level avg sessions/active-month: {person_avg_sessions['avg_sessions_per_active_month'].mean():.2f}")
print(f"SD of person-level avg sessions/active-month:   {person_avg_sessions['avg_sessions_per_active_month'].std():.2f}")

# ═════════════════════════════════════════════════════════════════════════════
# 4. Combined summary table
# ═════════════════════════════════════════════════════════════════════════════
summary_rows = {
    'N adopters (DiD sample)': f"{n_adopters:,}",
    'BMI, mean (SD)': f"{bmi_vals.mean():.2f} ({bmi_vals.std():.2f})",
    'BMI, median (IQR)': f"{bmi_vals.median():.1f} ({bmi_q1:.1f}\u2013{bmi_q3:.1f})",
    'BMI, missing, N (%)': f"{n_bmi_missing:,} ({100*n_bmi_missing/n_adopters:.1f}%)",
    'BMI \u2014 Underweight, N (%)': f"{cat_counts['Underweight (<18.5)']:,} ({cat_pct['Underweight (<18.5)']}%)",
    'BMI \u2014 Normal, N (%)':       f"{cat_counts['Normal (18.5-24.9)']:,} ({cat_pct['Normal (18.5-24.9)']}%)",
    'BMI \u2014 Overweight, N (%)':   f"{cat_counts['Overweight (25-29.9)']:,} ({cat_pct['Overweight (25-29.9)']}%)",
    'BMI \u2014 Obese, N (%)':        f"{cat_counts['Obese (>=30)']:,} ({cat_pct['Obese (>=30)']}%)",
    'Daily steps (person-level mean), mean (SD) across persons':
        f"{steps_per_person['mean_daily_steps'].mean():.1f} ({steps_per_person['mean_daily_steps'].std():.1f})",
    'Sessions per active month, mean (SD) across persons':
        f"{person_avg_sessions['avg_sessions_per_active_month'].mean():.2f} "
        f"({person_avg_sessions['avg_sessions_per_active_month'].std():.2f})",
}

summary_df = pd.DataFrame.from_dict(summary_rows, orient='index', columns=['Adopters (DiD sample)'])
print("\n=== Combined summary ===")
print(summary_df.to_string())

summary_df.to_csv('adopters_bmi_steps_sessions_summary.csv')
print("\nSaved: adopters_bmi_steps_sessions_summary.csv")

## Sample characteristics for adopters by adoption year

In [ ]:
# ── Full PCG label map (official, English) ────────────────────────────────────
pcg_labels_full = {
    'pcg_abh': 'Addiction (excl. Opioids)',
    'pcg_adh': 'ADHD',
    'pcg_aik': 'IBD',
    'pcg_alz': "Alzheimer's / Dementia",
    'pcg_ast': 'Asthma',
    'pcg_bsr': 'Bone / Skeletal Disorders',
    'pcg_car': 'Cardiac Arrhythmia',
    'pcg_cop': 'COPD',
    'pcg_dep': 'Depression',
    'pcg_dm1': 'Type 1 Diabetes',
    'pcg_dm2': 'Type 2 Diabetes',
    'pcg_dm2hyp': 'Type 2 Diabetes with Hypertension',  # combined flag
    'pcg_epi': 'Epilepsy',
    'pcg_gla': 'Glaucoma',
    'pcg_hch': 'High Cholesterol',
    'pcg_hiv': 'HIV',
    'pcg_kho': 'Coronary Heart Disease',
    'pcg_kre': 'Cancer',
    'pcg_krk': 'Cancer (other)',
    'pcg_mcr': 'Macular Degeneration',
    'pcg_msk': 'Musculoskeletal Disorders',
    'pcg_nie': 'Kidney Disease',
    'pcg_pah': 'Pulmonary Arterial Hypertension',
    'pcg_par': "Parkinson's",
    'pcg_pso': 'Psoriasis',
    'pcg_psy': 'Severe Psychiatric Disorders',
    'pcg_rhe': 'Rheumatoid Arthritis',
    'pcg_smc': 'Severe Mental Conditions',
    'pcg_smn': 'Neuropathic Pain',
    'pcg_thy': 'Thyroid Disorders',
    'pcg_tra': 'Transplantation',
    'pcg_was': 'Growth Disorders',
    'pcg_zfp': 'Cystic Fibrosis / Pancreatic Enzymes',
    'pcg_zns': 'CNS Disorders (excl. MS)',
}

# ── Full incidence table ───────────────────────────────────────────────────────
pcg_did['n_pcgs'] = pcg_did[pcg_cols].sum(axis=1)
pcg_pop['n_pcgs'] = pcg_pop[pcg_cols].sum(axis=1)

n_did_pcg, n_pop_pcg = n_did_rec, n_pop_rec  # people with ≥1 pcg_wide row
prev_did_all = (pcg_did[pcg_cols].sum() / n_did_pcg * 100).rename('DiD sample (%)')
prev_pop_all = (pcg_pop[pcg_cols].sum() / n_pop_pcg * 100).rename('Overall pop (%)')

df_all_pcgs = pd.concat([prev_did_all, prev_pop_all], axis=1).round(3)
df_all_pcgs['diff (DiD - pop)'] = (df_all_pcgs['DiD sample (%)'] - df_all_pcgs['Overall pop (%)']).round(3)
df_all_pcgs['pop_higher']       = df_all_pcgs['diff (DiD - pop)'] < 0
df_all_pcgs['label']            = df_all_pcgs.index.map(lambda x: pcg_labels_full.get(x.lower(), x))
df_all_pcgs['flag']             = df_all_pcgs['pop_higher'].map({True: '\u2b06 pop higher', False: ''})

df_all_pcgs = df_all_pcgs[['label', 'DiD sample (%)', 'Overall pop (%)', 'diff (DiD - pop)', 'pop_higher', 'flag']]
df_all_pcgs = df_all_pcgs.sort_values('Overall pop (%)', ascending=False)

print(f"=== PCG incidence (lifetime/ever-positive): DiD sample (N={n_did_pcg:,}) vs overall pop (N={n_pop_pcg:,}) ===\n")
print(df_all_pcgs.to_string())

n_pop_higher = df_all_pcgs['flag'].eq('\u2b06 pop higher').sum()
n_did_higher = (df_all_pcgs['flag'] == '').sum()
print(f"\nPCGs higher in DiD sample : {n_did_higher}")
print(f"PCGs higher in overall pop: {n_pop_higher}")

In [ ]:
# ── Bar chart: top 15 PCGs, labels coloured by higher-prevalence group ─────────

# Top 15 by overall population prevalence
top15 = df_all_pcgs.head(15).copy()
top15 = top15.sort_values('Overall pop (%)', ascending=True)  # ascending for horizontal bar

COLOR_DID = '#E84A5F'
COLOR_POP = '#2B4590'

fig, ax = plt.subplots(figsize=(12, 7))

y      = np.arange(len(top15))
height = 0.38

b1 = ax.barh(y + height/2, top15['DiD sample (%)'],
             height, color=COLOR_DID, alpha=0.75,
             label=f'DiD sample (N={n_did_pcg:,})')
b2 = ax.barh(y - height/2, top15['Overall pop (%)'],
             height, color=COLOR_POP, alpha=0.55,
             label=f'Overall population (N={n_pop_pcg:,})')

# Value labels on bars
for bar in b1:
    w = bar.get_width()
    if w > 0.05:
        ax.text(w + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{w:.1f}%', va='center', fontsize=8, color=COLOR_DID)
for bar in b2:
    w = bar.get_width()
    if w > 0.05:
        ax.text(w + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{w:.1f}%', va='center', fontsize=8, color=COLOR_POP)

# Y-tick labels coloured by which group has higher prevalence
ax.set_yticks(y)
ax.set_yticklabels([])  # clear first, then add manually with colour

for i, (_, row) in enumerate(top15.iterrows()):
    tick_color = COLOR_POP if row['pop_higher'] else COLOR_DID
    ax.text(-0.3, i, row['label'],
            ha='right', va='center', fontsize=10.5,
            color=tick_color, fontweight='bold',
            transform=ax.get_yaxis_transform())

ax.set_xlabel('Lifetime prevalence (%, ever-positive across 2018–2024)', fontsize=11)
ax.set_title(
    f'PCG lifetime prevalence: DiD sample (N={n_did_pcg:,}) vs. overall population (N={n_pop_pcg:,})\n'
    'Label colour indicates group with higher prevalence',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=10, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.savefig('pcg_prevalence_top15_did_vs_pop.png', dpi=150, bbox_inches='tight')
plt.show()

## Sample description diff-in-diff by adoption year

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Adopters/DiD sample characteristics table, stratified by adoption year
# (2020-2024), based on g_period_start (calendar adoption period) in mdid_input.
# ═════════════════════════════════════════════════════════════════════════════

# ── 0. Adoption year per person (from mdid_input, restricted to DiD sample) ────
# mdid_input has no g_period_start column directly; the adoption-quarter
# calendar date is the t_period_start of the row where event_time = 0
# (i.e. t_index == g_index).
con.execute("""
    CREATE OR REPLACE TEMP TABLE adoption_year AS
    SELECT DISTINCT
        m.eth_id,
        YEAR(CAST(m.t_period_start AS DATE)) AS adoption_year
    FROM mdid_input m
    INNER JOIN uid u ON u.eth_id = m.eth_id
    WHERE m.event_time = 0
      AND m.t_period_start IS NOT NULL
""")

print("Adoption year distribution (DiD sample):")
print(con.execute("""
    SELECT adoption_year, COUNT(*) AS n
    FROM adoption_year
    GROUP BY adoption_year
    ORDER BY adoption_year
""").fetchdf().to_string(index=False))

YEARS = [2020, 2021, 2022, 2023, 2024]

# ── 1. Per-person demographic/cost/claims/franchise/PCG table (DiD sample) ─────
did_full = con.execute("""
    WITH latest_person AS (
        SELECT DISTINCT ON (eth_id)
            eth_id,
            TRY_CAST(alter_kunde AS INTEGER) AS age,
            geschlecht,
            kanton
        FROM persons
        ORDER BY eth_id, TRY_CAST(endedatum AS DATE) DESC
    ),
    latest_income AS (
        SELECT DISTINCT ON (eth_id)
            eth_id, einkommensindex AS income_index
        FROM income_index
        ORDER BY eth_id, CAST(endedatum AS DATE) DESC
    ),
    latest_bmi AS (
        SELECT DISTINCT ON (eth_id)
            eth_id, bmi
        FROM claims_person_month_summary
        WHERE bmi IS NOT NULL
        ORDER BY eth_id, claim_month DESC
    )
    SELECT
        u.eth_id,
        lp.age,
        lp.geschlecht,
        lp.kanton,
        li.income_index,
        lb.bmi,
        cy.n_coverage_years,
        pc.n_inpatient_claims, pc.inpatient_cost,
        pc.n_outpatient_claims, pc.outpatient_cost,
        pc.total_claim_cost, pc.total_claims,
        pf.franchise
    FROM uid u
    LEFT JOIN latest_person lp ON lp.eth_id = u.eth_id
    LEFT JOIN latest_income li ON li.eth_id = u.eth_id
    LEFT JOIN latest_bmi lb ON lb.eth_id = u.eth_id
    LEFT JOIN coverage_years cy ON cy.eth_id = u.eth_id
    LEFT JOIN person_claims pc ON pc.eth_id = u.eth_id
    LEFT JOIN person_franchise pf ON pf.eth_id = u.eth_id
""").fetchdf()

did_full = did_full.merge(pcg_count_all, on='eth_id', how='left')
did_full['n_pcgs'] = did_full['n_pcgs'].fillna(0)

safe_years = did_full['n_coverage_years'].replace(0, np.nan)
did_full['avg_yearly_total_cost']        = did_full['total_claim_cost'] / safe_years
did_full['avg_yearly_inpatient_cost']    = did_full['inpatient_cost'] / safe_years
did_full['avg_yearly_outpatient_cost']   = did_full['outpatient_cost'] / safe_years
did_full['avg_yearly_total_claims']      = did_full['total_claims'] / safe_years
did_full['avg_yearly_inpatient_claims']  = did_full['n_inpatient_claims'] / safe_years
did_full['avg_yearly_outpatient_claims'] = did_full['n_outpatient_claims'] / safe_years

adoption_year_df = con.execute("SELECT * FROM adoption_year").fetchdf()
did_full = did_full.merge(adoption_year_df, on='eth_id', how='inner')  # drop anyone w/o a resolvable adoption year

# WHO BMI category
def who_category(b):
    if pd.isna(b):
        return 'Missing'
    elif b < 18.5:
        return 'Underweight'
    elif b < 25:
        return 'Normal'
    elif b < 30:
        return 'Overweight'
    else:
        return 'Obese'
did_full['who_cat'] = did_full['bmi'].apply(who_category)

# ── 2. Steps and sessions per person (reuse logic, all years, DiD sample) ──────
steps_per_person = con.execute("""
    SELECT
        a.eth_id,
        AVG(TRY_CAST(s.VALUE AS DOUBLE)) AS mean_daily_steps
    FROM uid a
    INNER JOIN a365_health_activities s
        ON CAST(s.CSS_MARKETING_ID AS VARCHAR) = CAST(a.eth_id AS VARCHAR)
    WHERE LOWER(TRIM(s.DATA_TYPE)) = 'steps'
      AND TRY_CAST(s.VALUE AS DOUBLE) IS NOT NULL
    GROUP BY a.eth_id
""").fetchdf()

sessions_pm = con.execute("""
    SELECT
        a.eth_id,
        DATE_TRUNC('month', CAST(s.DATE_TIME AS TIMESTAMP)) AS session_month,
        COUNT(*) AS n_sessions
    FROM uid a
    INNER JOIN a365_health_sessions s
        ON CAST(s.CSS_MARKETING_ID AS VARCHAR) = CAST(a.eth_id AS VARCHAR)
    WHERE s.DATE_TIME IS NOT NULL
    GROUP BY a.eth_id, DATE_TRUNC('month', CAST(s.DATE_TIME AS TIMESTAMP))
""").fetchdf()
person_avg_sessions = (
    sessions_pm.groupby('eth_id')['n_sessions'].mean()
    .reset_index(name='avg_sessions_per_active_month')
)
person_any_session = sessions_pm[['eth_id']].drop_duplicates()
person_any_session['has_session'] = 1

did_full = did_full.merge(steps_per_person, on='eth_id', how='left')
did_full = did_full.merge(person_avg_sessions, on='eth_id', how='left')
did_full = did_full.merge(person_any_session[['eth_id', 'has_session']], on='eth_id', how='left')
did_full['has_session'] = did_full['has_session'].fillna(0)
did_full['has_step_day'] = did_full['mean_daily_steps'].notna().astype(int)

# ── 3. Observation time (person-quarters, pre/post) from mdid_input ────────────
obs_time = con.execute("""
    SELECT
        m.eth_id,
        ay.adoption_year,
        COUNT(*) AS n_quarters,
        COUNT(*) FILTER (WHERE m.event_time < 0) AS n_pre_quarters,
        COUNT(*) FILTER (WHERE m.event_time >= 0) AS n_post_quarters
    FROM mdid_input m
    INNER JOIN adoption_year ay ON ay.eth_id = m.eth_id
    GROUP BY m.eth_id, ay.adoption_year
""").fetchdf()

# ── 4. Formatting helpers ───────────────────────────────────────────────────────
def fmt_mean_sd(s):
    s = s.dropna()
    return f"{s.mean():.2f} ({s.std():.2f})" if len(s) else "n/a"

def fmt_median_range(s):
    s = s.dropna()
    return f"{s.median():.1f} ({s.min():.0f}\u2013{s.max():.0f})" if len(s) else "n/a"

def fmt_median_iqr(s):
    s = s.dropna()
    if not len(s):
        return "n/a"
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return f"{s.median():.2f} ({q1:.2f} to {q3:.2f})"

def pct(n, d):
    return f"{100*n/d:.1f}%" if d else "n/a"

# ── 5. Build summary for one adoption-year slice ────────────────────────────────
def summarize_year(yr):
    tbl = did_full[did_full['adoption_year'] == yr]
    n = len(tbl)
    obs = obs_time[obs_time['adoption_year'] == yr]

    rows = {}
    rows['N adopters'] = f"{n:,}"

    rows['Age, mean (SD)']          = fmt_mean_sd(tbl['age'])
    rows['Age, median (range)']     = fmt_median_range(tbl['age'])

    n_f = (tbl['geschlecht'] == 'F').sum()
    n_m = (tbl['geschlecht'] == 'M').sum()
    n_other = n - n_f - n_m
    rows['Sex, female']             = pct(n_f, n)
    rows['Sex, male']               = pct(n_m, n)
    rows['Sex, other/missing']      = pct(n_other, n)

    rows['Income index, median (IQR)'] = fmt_median_iqr(tbl['income_index'])

    rows['Avg yearly total cost, mean (SD)']      = fmt_mean_sd(tbl['avg_yearly_total_cost'])
    rows['Avg yearly inpatient cost, mean (SD)']  = fmt_mean_sd(tbl['avg_yearly_inpatient_cost'])
    rows['Avg yearly outpatient cost, mean (SD)'] = fmt_mean_sd(tbl['avg_yearly_outpatient_cost'])

    rows['Avg yearly total claims, mean (SD)']      = fmt_mean_sd(tbl['avg_yearly_total_claims'])
    rows['Avg yearly inpatient claims, mean (SD)']  = fmt_mean_sd(tbl['avg_yearly_inpatient_claims'])
    rows['Avg yearly outpatient claims, mean (SD)'] = fmt_mean_sd(tbl['avg_yearly_outpatient_claims'])

    rows['Franchise, mean (SD)'] = fmt_mean_sd(tbl['franchise'])
    rows['Franchise, median']    = f"{tbl['franchise'].median():.0f}" if tbl['franchise'].notna().any() else "n/a"

    rows['PCGs, mean (SD)']  = fmt_mean_sd(tbl['n_pcgs'])
    rows['PCGs, median']     = f"{tbl['n_pcgs'].median():.1f}"
    rows['\u22651 PCG']           = pct((tbl['n_pcgs'] >= 1).sum(), n)

    bmi_vals = tbl['bmi'].dropna()
    n_bmi_missing = tbl['bmi'].isna().sum()
    rows['BMI, mean (SD)']        = fmt_mean_sd(tbl['bmi'])
    rows['BMI, median (IQR)']     = fmt_median_iqr(tbl['bmi'])
    rows['BMI, missing']          = f"{n_bmi_missing:,} ({100*n_bmi_missing/n:.1f}%)" if n else "n/a"

    cat_order = ['Underweight', 'Normal', 'Overweight', 'Obese']
    cat_counts = tbl['who_cat'].value_counts().reindex(cat_order, fill_value=0)
    for cat in cat_order:
        rows[f'BMI category \u2014 {cat}'] = f"{cat_counts[cat]:,} ({100*cat_counts[cat]/n:.1f}%)" if n else "n/a"

    n_step_day = tbl['has_step_day'].sum()
    rows['Any recorded step-day']           = f"{n_step_day:,} ({100*n_step_day/n:.1f}%)" if n else "n/a"
    rows['Daily steps, person-mean (SD)']   = fmt_mean_sd(tbl['mean_daily_steps'])

    n_any_session = tbl['has_session'].sum()
    rows['Any session']                       = f"{n_any_session:,} ({100*n_any_session/n:.1f}%)" if n else "n/a"
    rows['Sessions/active month, mean (SD)'] = fmt_mean_sd(tbl['avg_sessions_per_active_month'])

    rows['Person-quarters']                    = f"{obs['n_quarters'].sum():,}"
    rows['Quarters/person, mean']              = f"{obs['n_quarters'].mean():.2f}" if len(obs) else "n/a"
    rows['Quarters/person, median (range)']    = fmt_median_range(obs['n_quarters'])
    rows['Pre-adoption quarters, mean']        = f"{obs['n_pre_quarters'].mean():.2f}" if len(obs) else "n/a"
    rows['Pre-adoption quarters, median (range)']  = fmt_median_range(obs['n_pre_quarters'])
    rows['Post-adoption quarters, mean']       = f"{obs['n_post_quarters'].mean():.2f}" if len(obs) else "n/a"
    rows['Post-adoption quarters, median (range)'] = fmt_median_range(obs['n_post_quarters'])

    return rows

# ── 6. Assemble combined by-year table ──────────────────────────────────────────
year_summaries = {str(yr): summarize_year(yr) for yr in YEARS}
by_year_table = pd.DataFrame(year_summaries)

print("\n=== Adopters characteristics by adoption year ===")
print(by_year_table.to_string())

by_year_table.to_csv('did_sample_characteristics_by_adoption_year.csv')
print("\nSaved: did_sample_characteristics_by_adoption_year.csv")

# Insurance claims volume (cost and number of claims) by year for the general population

In [ ]:
# Legal bases ('gesetz'): claims count and volume by year, FILTERED GENERAL POPULATION ──
# Restricted to filtered_pop (18+, ever held a mandatory/contracts_obligatory contract).

gesetz_table_filtered = con.execute("""
    WITH combined AS (
        SELECT
            'inpatient'                                                    AS source,
            COALESCE(NULLIF(TRIM(i.gesetz), ''), '(missing)')              AS gesetz,
            EXTRACT(YEAR FROM TRY_CAST(i.behandlungsbeginn AS DATE))::INT  AS year,
            COUNT(*)                                                       AS n_claims,
            SUM(TRY_CAST(i.betrag AS DOUBLE))                              AS total_cost
        FROM inpatient i
        INNER JOIN filtered_pop fp ON fp.eth_id = i.eth_id
        WHERE i.behandlungsbeginn IS NOT NULL
        GROUP BY 1, 2, 3

        UNION ALL

        SELECT
            'outpatient'                                                   AS source,
            COALESCE(NULLIF(TRIM(o.gesetz), ''), '(missing)')              AS gesetz,
            EXTRACT(YEAR FROM TRY_CAST(o.behandlungsbeginn AS DATE))::INT  AS year,
            COUNT(*)                                                       AS n_claims,
            SUM(TRY_CAST(o.betrag AS DOUBLE))                              AS total_cost
        FROM outpatient o
        INNER JOIN filtered_pop fp ON fp.eth_id = o.eth_id
        WHERE o.behandlungsbeginn IS NOT NULL
        GROUP BY 1, 2, 3
    )
    SELECT source, gesetz, year, n_claims, ROUND(total_cost, 2) AS total_cost_chf
    FROM combined
    ORDER BY source, gesetz, year
""").fetchdf()

print("=== Claims count and volume by source / gesetz / year (filtered general population) ===")
print(gesetz_table_filtered.to_string(index=False))

# ═════════════════════════════════════════════════════════════════════════════
# Wide table: rows = source × legal basis × metric, columns = years + Total
# ═════════════════════════════════════════════════════════════════════════════
years = [2018, 2019, 2020, 2021, 2022, 2023, 2024]
gesetz_order = ['KVG', 'VVG', 'UVG', 'IVG', 'MVG', 'OR', 'unbekannt', '(missing)']

df = gesetz_table_filtered[gesetz_table_filtered['year'].isin(years)]

wide = (df.melt(id_vars=['source', 'gesetz', 'year'],
                value_vars=['n_claims', 'total_cost_chf'],
                var_name='metric', value_name='value')
          .pivot_table(index=['source', 'gesetz', 'metric'], columns='year',
                       values='value', aggfunc='sum', fill_value=0)
          .reindex(columns=years, fill_value=0))
wide['Total'] = wide.sum(axis=1)
wide = wide.rename(index={'n_claims': 'n', 'total_cost_chf': 'CHF'}, level='metric')

# canonical order; any unexpected gesetz values go last instead of being dropped
order = {g: k for k, g in enumerate(gesetz_order)}
wide = wide.reset_index()
wide['_o'] = wide['gesetz'].map(order).fillna(len(order))
wide = (wide.sort_values(['source', '_o', 'gesetz', 'metric'], ascending=[True, True, True, False])
            .drop(columns='_o')
            .rename(columns={'source': 'Source', 'gesetz': 'Legal basis', 'metric': 'Metric'})
            .set_index(['Source', 'Legal basis', 'Metric']))

# Save raw numbers; print a readable version
wide.to_csv('results/descriptives/gesetz_wide_filtered_pop.csv')

def fmt(v, metric):
    if v == 0:
        return '—'
    if metric == 'n':
        return f"{int(v):,}"
    m = v / 1e6
    return f"{m/1000:.1f}B" if m >= 1000 else f"{m:.1f}M" if m >= 1 else f"{m*1000:.0f}K"

wide_fmt = wide.copy().astype(object)
for idx in wide.index:
    wide_fmt.loc[idx] = [fmt(v, idx[2]) for v in wide.loc[idx]]

print("\n=== Claims count (n) and volume (CHF) by legal basis, source and year "
      "(filtered general population: age 18+, ever held a mandatory contract) ===")
print(wide_fmt.to_string())
print("\nSaved: results/descriptives/gesetz_wide_filtered_pop.csv")

# Missingness by variable

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Missingness table: all characteristics variables, by adoption status
# (DiD sample / adopters  vs.  filtered general population)
# ═════════════════════════════════════════════════════════════════════════════

def build_group_table_full(eth_ids_query):
    """Rebuild the full per-person variable set for a given group (eth_id list)."""
    demo = con.execute(f"""
        WITH grp AS ({eth_ids_query}),
        latest_person AS (
            SELECT DISTINCT ON (eth_id)
                eth_id,
                TRY_CAST(alter_kunde AS INTEGER) AS age,
                geschlecht,
                kanton
            FROM persons
            ORDER BY eth_id, TRY_CAST(endedatum AS DATE) DESC
        ),
        latest_income AS (
            SELECT DISTINCT ON (eth_id)
                eth_id, einkommensindex AS income_index
            FROM income_index
            ORDER BY eth_id, CAST(endedatum AS DATE) DESC
        ),
        latest_bmi AS (
            SELECT DISTINCT ON (eth_id)
                eth_id, bmi
            FROM claims_person_month_summary
            WHERE bmi IS NOT NULL
            ORDER BY eth_id, claim_month DESC
        )
        SELECT
            g.eth_id,
            lp.age,
            lp.geschlecht,
            lp.kanton,
            li.income_index,
            lb.bmi,
            cy.n_coverage_years,
            pc.n_inpatient_claims, pc.inpatient_cost,
            pc.n_outpatient_claims, pc.outpatient_cost,
            pc.total_claim_cost, pc.total_claims,
            pf.franchise
        FROM grp g
        LEFT JOIN latest_person lp ON lp.eth_id = g.eth_id
        LEFT JOIN latest_income li ON li.eth_id = g.eth_id
        LEFT JOIN latest_bmi lb ON lb.eth_id = g.eth_id
        LEFT JOIN coverage_years cy ON cy.eth_id = g.eth_id
        LEFT JOIN person_claims pc ON pc.eth_id = g.eth_id
        LEFT JOIN person_franchise pf ON pf.eth_id = g.eth_id
    """).fetchdf()

    demo = demo.merge(pcg_count_all, on='eth_id', how='left')
    # post-merge; treat unmatched persons as missing here (NaN preserved, not filled).

    safe_years = demo['n_coverage_years'].replace(0, np.nan)
    demo['avg_yearly_total_cost']        = demo['total_claim_cost'] / safe_years
    demo['avg_yearly_inpatient_cost']    = demo['inpatient_cost'] / safe_years
    demo['avg_yearly_outpatient_cost']   = demo['outpatient_cost'] / safe_years
    demo['avg_yearly_total_claims']      = demo['total_claims'] / safe_years
    demo['avg_yearly_inpatient_claims']  = demo['n_inpatient_claims'] / safe_years
    demo['avg_yearly_outpatient_claims'] = demo['n_outpatient_claims'] / safe_years

    # Steps and sessions, computed fresh for this group (eth_id == CSS_MARKETING_ID)
    steps = con.execute(f"""
        SELECT
            g.eth_id,
            AVG(TRY_CAST(s.VALUE AS DOUBLE)) AS mean_daily_steps
        FROM ({eth_ids_query}) g
        INNER JOIN a365_health_activities s
            ON CAST(s.CSS_MARKETING_ID AS VARCHAR) = CAST(g.eth_id AS VARCHAR)
        WHERE LOWER(TRIM(s.DATA_TYPE)) = 'steps'
          AND TRY_CAST(s.VALUE AS DOUBLE) IS NOT NULL
        GROUP BY g.eth_id
    """).fetchdf()

    sessions_pm = con.execute(f"""
        SELECT
            g.eth_id,
            DATE_TRUNC('month', CAST(s.DATE_TIME AS TIMESTAMP)) AS session_month,
            COUNT(*) AS n_sessions
        FROM ({eth_ids_query}) g
        INNER JOIN a365_health_sessions s
            ON CAST(s.CSS_MARKETING_ID AS VARCHAR) = CAST(g.eth_id AS VARCHAR)
        WHERE s.DATE_TIME IS NOT NULL
        GROUP BY g.eth_id, DATE_TRUNC('month', CAST(s.DATE_TIME AS TIMESTAMP))
    """).fetchdf()
    person_avg_sessions = (
        sessions_pm.groupby('eth_id')['n_sessions'].mean()
        .reset_index(name='avg_sessions_per_active_month')
    )

    demo = demo.merge(steps, on='eth_id', how='left')
    demo = demo.merge(person_avg_sessions, on='eth_id', how='left')

    return demo

did_full_chars = build_group_table_full("SELECT eth_id FROM uid")
pop_full_chars = build_group_table_full("SELECT eth_id FROM filtered_pop")

n_did = len(did_full_chars)
n_pop = len(pop_full_chars)

# ── vars list to readable labels ───────────────────────────────────
variables = {
    'age': 'Age',
    'geschlecht': 'Sex',
    'kanton': 'Canton',
    'income_index': 'Income index',
    'avg_yearly_total_cost': 'Avg yearly total cost',
    'avg_yearly_inpatient_cost': 'Avg yearly inpatient cost',
    'avg_yearly_outpatient_cost': 'Avg yearly outpatient cost',
    'avg_yearly_total_claims': 'Avg yearly total claims',
    'avg_yearly_inpatient_claims': 'Avg yearly inpatient claims',
    'avg_yearly_outpatient_claims': 'Avg yearly outpatient claims',
    'franchise': 'Franchise',
    'n_pcgs': 'Number of PCGs',
    'bmi': 'BMI',
    'mean_daily_steps': 'Daily steps (person-mean)',
    'avg_sessions_per_active_month': 'Sessions per active month',
}

def missingness_row(tbl, n_total, col):
    n_missing = tbl[col].isna().sum()
    pct_missing = 100 * n_missing / n_total if n_total else float('nan')
    return n_missing, pct_missing

rows = []
for col, label in variables.items():
    n_miss_did, pct_miss_did = missingness_row(did_full_chars, n_did, col)
    n_miss_pop, pct_miss_pop = missingness_row(pop_full_chars, n_pop, col)
    rows.append({
        'Variable': label,
        f'DiD sample (N={n_did:,}) \u2014 missing N': n_miss_did,
        f'DiD sample (N={n_did:,}) \u2014 missing %': round(pct_miss_did, 1),
        f'Overall pop (N={n_pop:,}) \u2014 missing N': n_miss_pop,
        f'Overall pop (N={n_pop:,}) \u2014 missing %': round(pct_miss_pop, 1),
    })

missingness_table = pd.DataFrame(rows).set_index('Variable')

print("=== Missingness by variable and adoption status ===\n")
print(missingness_table.to_string())

missingness_table.to_csv('results/descriptives/missingness_by_adoption_status.csv')
print("\nSaved: missingness_by_adoption_status.csv")

In [ ]:
con.close()